In [7]:
import pyodbc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import credentials as creds

# 1. CONNECT
conn_str = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={creds.SERVER};"
    f"DATABASE={creds.DATABASE};"
    f"UID={creds.USER};"
    f"PWD={creds.PASSWORD};"
    "Encrypt=no;"
    "TrustServerCertificate=yes;"
)

conn = pyodbc.connect(conn_str)

# 2. THE ANALYST QUERY (Invoice Rows joined to Headers)
# We join OINV (Header) to INV1 (Rows) to get Sales per Item
sql = """
SELECT TOP 10
    Items.ItemCode,
    Items.Dscription,
    SUM(Items.LineTotal) as NetRevenue
FROM (
    -- 1. Get All Invoices (Positives)
    SELECT T1.ItemCode, T1.Dscription, T1.LineTotal
    FROM OINV T0
    INNER JOIN INV1 T1 ON T0.DocEntry = T1.DocEntry
    WHERE T0.DocDate >= '2024-01-01' AND T0.CANCELED = 'N'

    UNION ALL

    -- 2. Get All Credit Memos (Negatives)
    SELECT T1.ItemCode, T1.Dscription, -T1.LineTotal -- Note the negative sign
    FROM ORIN T0
    INNER JOIN RIN1 T1 ON T0.DocEntry = T1.DocEntry
    WHERE T0.DocDate >= '2024-01-01' AND T0.CANCELED = 'N'
) AS Items
GROUP BY Items.ItemCode, Items.Dscription
ORDER BY NetRevenue DESC
"""

# 3. GET DATA
df = pd.read_sql(sql, conn)
conn.close()

# 4. VISUALIZE
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

# Create Bar Chart
chart = sns.barplot(
    data=df,
    x="TotalRevenue",
    y="Dscription",
    palette="viridis"
)

plt.title("🏆 Top 10 Best-Selling Items (Revenue)", fontsize=16)
plt.xlabel("Revenue (PHP)", fontsize=12)
plt.ylabel("Item Name", fontsize=12)
plt.show()

InterfaceError: ('28000', "[28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Login failed for user 'sa'. (18456) (SQLDriverConnect)")